## Imports and Reading Data

In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import statsmodels.api as sm
from patsy import dmatrices
from scipy.stats import norm
# ignore futurewarning
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
customers = pd.read_csv('../data/raw/customers.csv')
responses = pd.read_csv('../data/raw/responses.csv')
views = pd.read_csv('../data/raw/views.csv')

In [5]:
responses.sample(5)

,customer_id,segment,offer,resubscribed_promo,resub_day
334107,334108,Medium,Early Access,0,NaN
91377,91378,High,Free Trial,0,NaN
111003,111004,Low,Control,0,NaN
389326,389327,Low,Free Trial,0,NaN
336880,336881,Low,Control,0,NaN


## Analysis 1 - Campaign Incrementality

In [6]:
inc_analysis_df = responses.copy()
inc_analysis_df.sample(5)

,customer_id,segment,offer,resubscribed_promo,resub_day
22569,22570,Low,Free Trial,0,NaN
36828,36829,Low,Discount,1,9.0
382930,382931,Low,Free Trial,0,NaN
425367,425368,Medium,Free Trial,0,NaN
147620,147621,Low,Early Access,0,NaN


In [11]:
inc_analysis_df['control'] = inc_analysis_df['offer'].apply(lambda x: 1 if x == 'Control' else 0)
inc_analysis_df['control'].value_counts()

control
0    450002
1     49998
Name: count, dtype: int64

In [18]:
formula = (
    'resubscribed_promo ~ control * segment'
)
y, X = dmatrices(formula, inc_analysis_df, return_type='dataframe')

model = sm.Logit(y, X).fit()

Optimization terminated successfully.
         Current function value: 0.443810
         Iterations 7


In [20]:
# Check the profile of the response variable (resubscribed_promo)
resp_profile = inc_analysis_df[model.model.endog_names].value_counts().sort_index(ascending=False)

resp_profile_df = pd.DataFrame({
    'Ordered Value': range(1, len(resp_profile) + 1),
    model.model.endog_names: resp_profile.index,
    'Total': resp_profile.values
})

resp_profile_df

,Ordered Value,resubscribed_promo,Total
0,1,1,83656
1,2,0,416344


In [22]:
# model outputs
summary_df = pd.DataFrame({
    'variable': model.params.index,
    'coef': model.params.values,
    'std err': model.bse.values,
    'z': model.tvalues.values,
    'p-value': model.pvalues.values
})
summary_df

,variable,coef,std err,z,p-value
0,Intercept,-1.072979,0.008833,-121.480490,0.000000e+00
1,segment[T.Low],-0.487781,0.010442,-46.714187,0.000000e+00
2,segment[T.Medium],-0.652745,0.011289,-57.822497,0.000000e+00
3,control,-1.138314,0.039590,-28.752698,8.380820e-182
4,control:segment[T.Low],-0.066203,0.048197,-1.373581,1.695718e-01
5,control:segment[T.Medium],0.407937,0.048928,8.337424,7.592613e-17


In [23]:
X.columns

Index(['Intercept', 'segment[T.Low]', 'segment[T.Medium]', 'control',
       'control:segment[T.Low]', 'control:segment[T.Medium]'],
      dtype='object')

In [28]:
# Get all parameter names
param_names = X.columns

# Get number of levels for categorical variables
n_control = inc_analysis_df['control'].nunique()
n_segment = inc_analysis_df['segment'].nunique()

#Main Columns
control_cols = [col for col in param_names if col.startswith('control') and ':' not in col]
segment_cols = [col for col in param_names if col.startswith('segment') and ':' not in col]

# Interaction Columns
interaction_cols = [col for col in param_names if ':' in col]

# Calculate DF
df_control = n_control - 1
df_segment = n_segment - 1
df_interaction = df_control * df_segment

#Results
joint_tests = []

if control_cols:
    test = model.wald_test(np.eye(len(param_names))[ [param_names.get_loc(col) for col in control_cols]])
    joint_tests.append(['control', df_control, test.statistic[0][0], test.pvalue])

if segment_cols:
    test = model.wald_test(np.eye(len(param_names))[ [param_names.get_loc(col) for col in segment_cols]])
    joint_tests.append(['segment', df_segment, test.statistic[0][0], test.pvalue])

if interaction_cols:
    test = model.wald_test(np.eye(len(param_names))[ [param_names.get_loc(col) for col in interaction_cols]])
    joint_tests.append(['interaction', df_interaction, test.statistic[0][0], test.pvalue])

joint_df = pd.DataFrame(joint_tests, columns=['Type', 'DF', 'Wald-Chisq Statistic', 'P-Value'])
joint_df['Significant'] = joint_df['P-Value'] < 0.01

joint_df

,Type,DF,Wald-Chisq Statistic,P-Value,Significant
0,control,1,826.717625,8.380819676848794e-182,True
1,segment,2,3470.359767,0.0,True
2,interaction,2,155.166368,2.023257668407754e-34,True


In [30]:
X.columns

Index(['Intercept', 'segment[T.Low]', 'segment[T.Medium]', 'control',
       'control:segment[T.Low]', 'control:segment[T.Medium]'],
      dtype='object')

In [33]:
contrast_rows = {
    'Control High':{
        'Intercept':1,
        'segment[T.Low]':0,
        'segment[T.Medium]':0,
        'control':1,
        'control:segment[T.Low]':0,
        'control:segment[T.Medium]':0
    },
    'Control Medium':{
        'Intercept':1,
        'segment[T.Low]':0,
        'segment[T.Medium]':1,
        'control':1,
        'control:segment[T.Low]':0,
        'control:segment[T.Medium]':1
    },
    'Control Low':{
        'Intercept':1,
        'segment[T.Low]':1,
        'segment[T.Medium]':0,
        'control':1,
        'control:segment[T.Low]':1,
        'control:segment[T.Medium]':0
    },
    'Offer High':{
        'Intercept':1,
        'segment[T.Low]':0,
        'segment[T.Medium]':0,
        'control':0,
        'control:segment[T.Low]':0,
        'control:segment[T.Medium]':0
    },
    'Offer Medium':{
        'Intercept':1,
        'segment[T.Low]':0,
        'segment[T.Medium]':1,
        'control':0,
        'control:segment[T.Low]':0,
        'control:segment[T.Medium]':0
    },
    'Offer Low':{
        'Intercept':1,
        'segment[T.Low]':1,
        'segment[T.Medium]':0,
        'control':0,
        'control:segment[T.Low]':0,
        'control:segment[T.Medium]':0
    },    
}

predicted_probs = {}
for label, row in contrast_rows.items():
    contrast_X = pd.DataFrame([row], columns=X.columns)
    pred = model.get_prediction(contrast_X)
    summary = pred.summary_frame(alpha=0.1)
    predicted_probs[label] = {
        'predicted': summary['predicted'][0],
        'se': summary['se'][0],
        'ci_lower': summary['ci_lower'][0],
        'ci_upper': summary['ci_upper'][0]
    }

predicted_df = pd.DataFrame(predicted_probs).T.reset_index().rename(columns={'index':'Contrast'})
predicted_df

,Contrast,predicted,se,ci_lower,ci_upper
0,Control High,0.098741,0.003434,0.093234,0.104536
1,Control Medium,0.078993,0.002028,0.075721,0.082395
2,Control Low,0.059230,0.001500,0.056810,0.061745
3,Offer High,0.254837,0.001677,0.252088,0.257606
4,Offer Medium,0.151135,0.000902,0.149658,0.152625
5,Offer Low,0.173538,0.000799,0.172228,0.174855


Incremental Factors and significance for each segment

In [37]:
params = model.params
cov = model.cov_params()

contrast_vectors = {'High Offer - Control': np.array([0, 0, 0, -1, 0, 0]),
                    'Medium Offer - Control': np.array([0, 0, 0, -1, 0, -1]),
                    'Low Offer - Control': np.array([0, 0, 0, -1, -1, 0])
}

# Z critical value for 95% confidence
z_crit = norm.ppf(0.975)

# Number of comparisons for Bonferroni correction
k = len(contrast_vectors)

rows = []
for label, contrast in contrast_vectors.items():
    est = np.dot(contrast, params)
    se = np.sqrt(np.dot(contrast.T, np.dot(cov, contrast)))
    z_val = est / se
    p_val = 2 * (1 - norm.cdf(abs(z_val)))
    
    #Bonferroni correction
    p_val_bonf = min(p_val * k, 1)
    
    #Seheffe adjustment 
    p_val_scheffe = 1 - (1 - p_val) ** k
    
    lower = est - z_crit * se
    upper = est + z_crit * se
    exp_est = np.exp(est)
    exp_lower = np.exp(lower)
    exp_upper = np.exp(upper)
    lift = round((exp_est - 1)*100)
    rows.append([label, est, se, z_val, p_val, p_val_bonf, p_val_scheffe, lower, upper, exp_est, exp_lower, exp_upper, lift])

result_df = pd.DataFrame(rows, columns=[
    'Contrast', 
    'Estimate',
    'SE',
    'Z-value',
    'P-value',
    'P-value (Bonf)',
    'P-value (Scheffe)',
    'CI Lower',
    'CI Upper',
    'Exp(Estimate)',
    'Exp(CI Lower)',
    'Exp(CI Upper)',
    'Lift (%)'
    ])

result_df

,Contrast,Estimate,SE,Z-value,P-value,P-value (Bonf),P-value (Scheffe),CI Lower,CI Upper,Exp(Estimate),Exp(CI Lower),Exp(CI Upper),Lift (%)
0,High Offer - Control,1.138314,0.039590,28.752698,0.0,0.0,0.0,1.060720,1.215909,3.121502,2.888449,3.373359,212
1,Medium Offer - Control,0.730377,0.028751,25.403271,0.0,0.0,0.0,0.674026,0.786729,2.075863,1.962120,2.196200,108
2,Low Offer - Control,1.204517,0.027488,43.819510,0.0,0.0,0.0,1.150641,1.258393,3.335147,3.160218,3.519759,234


## Analysis 2 - Offer Performance by Segment

## Analysis 3 - Post-Resubscription Engagement